# EP06. 에이전트 디버깅 & 모니터링

## 학습 목표

1. LangGraph 에이전트에 **Langfuse 트레이싱**을 연동한다
2. **무한루프 / 환각 / 도구 오용** 패턴을 재현하고 탐지한다
3. `recursion_limit`, `interrupt()`, `fallback`으로 **가드레일**을 구현한다
4. Langfuse **score API**로 에이전트 품질을 정량화한다

## AI 가이드

> 이 노트북을 학습하며 막히는 부분은 Claude에게 물어보세요.
> 예시 프롬프트:
> - "Langfuse CallbackHandler와 LangGraph를 연결하는 방법을 설명해줘"
> - "GraphRecursionError가 발생했을 때 어떻게 처리해야 해?"
> - "Langfuse score API로 환각 점수를 기록하는 코드 예시를 보여줘"

**사전 요구사항**: EP04 (기본 LangGraph), EP05 (에이전트 평가 기초)

**예상 소요 시간**: 90분

**필요한 API 키**:
- `ANTHROPIC_API_KEY`
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`

## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langchain langgraph langchain-anthropic langfuse python-dotenv

## 섹션 2. 라이브러리 로드 + Langfuse 초기화

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 확인 (직접 설정하거나 .env 파일에서 로드)
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode
from langgraph.errors import GraphRecursionError
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

# Langfuse 초기화
langfuse = Langfuse(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
)

# 연결 확인
try:
    langfuse.auth_check()
    print("Langfuse 연결 성공!")
except Exception as e:
    print(f"Langfuse 연결 실패 (노트북 학습은 계속 가능): {e}")

# LLM 초기화
llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
print("라이브러리 로드 완료!")

## 섹션 3. 기본 LangGraph 에이전트 구축

간단한 검색 에이전트를 만들어 Langfuse 연동의 베이스라인으로 사용합니다.

In [ ]:
# 도구 정의
@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색합니다. 실시간 정보가 필요할 때 사용하세요."""
    # 실제 검색 대신 시뮬레이션
    results = {
        "파이썬": "파이썬은 인터프리터 언어로 배우기 쉬운 범용 프로그래밍 언어입니다.",
        "LangChain": "LangChain은 LLM 애플리케이션 구축을 위한 프레임워크입니다.",
    }
    for key, value in results.items():
        if key in query:
            return value
    return f"'{query}'에 대한 검색 결과를 찾지 못했습니다."

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다. 예: '2 + 2', '10 * 5'"""
    try:
        # 안전한 계산만 허용
        allowed = set('0123456789+-*/()., ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return str(result)
        return "안전하지 않은 계산식입니다."
    except Exception as e:
        return f"계산 오류: {e}"

tools = [search_web, calculate]
llm_with_tools = llm.bind_tools(tools)

# LangGraph 에이전트 정의
def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 그래프 구성
builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.set_entry_point("agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")

basic_agent = builder.compile()
print("기본 에이전트 구축 완료!")

## 섹션 4. Langfuse CallbackHandler 통합 및 트레이스 확인

**[Langfuse 트레이싱 아키텍처]**
```mermaid
flowchart LR
    A((👤 사용자 쿼리)):::user --> B{"🤖 LangGraph 에이전트"}:::agent
    B --> C("🛠️ 도구 호출"):::tool
    C --> D("⚡ 도구 실행"):::tool
    D --> B
    B --> E([✨ 최종 응답]):::end
    B -. 비동기 트레이스 .-> F[/"📡 Langfuse SDK"\]:::sdk
    C -. span .-> F
    D -. span .-> F
    F --> G[("☁️ Langfuse 서버")]:::server
    G --> H("📊 대시보드 시각화"):::dash
    G --> I("⚖️ LLM-as-Judge 평가"):::dash
    G --> J("🚨 알림 / 경보"):::dash
    classDef user fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef agent fill:#fff3e0,stroke:#fb8c00,stroke-width:2px,color:#000
    classDef tool fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px,color:#000
    classDef end fill:#e8f5e9,stroke:#43a047,stroke-width:2px,color:#000
    classDef sdk fill:#e0f7fa,stroke:#00acc1,stroke-width:2px,color:#000
    classDef server fill:#eceff1,stroke:#607d8b,stroke-width:2px,color:#000
    classDef dash fill:#ffecb3,stroke:#ffb300,stroke-width:2px,color:#000
```

`CallbackHandler`를 주입하면 모든 LLM 호출과 도구 실행이 자동으로 Langfuse에 기록됩니다.

In [ ]:
# Langfuse CallbackHandler 생성
langfuse_handler = CallbackHandler(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
    session_id="ep06-basic-test",
    user_id="student-001",
    tags=["ep06", "debugging", "basic-test"],
)

# 에이전트 실행 (CallbackHandler 주입)
result = basic_agent.invoke(
    {"messages": [HumanMessage(content="파이썬이 무엇인지 검색하고 설명해줘")]},
    config={
        "callbacks": [langfuse_handler],
        "recursion_limit": 10,
    },
)

print("=== 에이전트 응답 ===")
print(result["messages"][-1].content)
print("\nLangfuse 대시보드에서 트레이스를 확인하세요!")
print(f"URL: {os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")

## 섹션 5. 무한루프 시나리오 생성

의도적으로 무한루프를 유발하는 에이전트를 만들어 문제를 재현합니다.

**루프 조건**: 도구가 항상 "다시 검색해야 합니다"를 반환하여 에이전트가 계속 검색을 시도

In [ ]:
# 무한루프를 유발하는 도구 (항상 불완전한 결과 반환)
call_count = {"search": 0}

@tool
def looping_search(query: str) -> str:
    """검색을 수행합니다. 결과가 불완전하면 다시 검색해야 합니다."""
    call_count["search"] += 1
    print(f"  [도구 호출 #{call_count['search']}] 검색어: '{query}'")
    # 항상 불완전한 결과를 반환하여 에이전트가 재검색하도록 유도
    return "결과가 불완전합니다. 더 구체적인 검색어로 다시 검색해주세요."

# 무한루프 유발 에이전트
looping_llm = llm.bind_tools([looping_search])

def looping_agent_node(state: MessagesState):
    response = looping_llm.invoke(state["messages"])
    return {"messages": [response]}

def looping_should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

loop_builder = StateGraph(MessagesState)
loop_builder.add_node("agent", looping_agent_node)
loop_builder.add_node("tools", ToolNode([looping_search]))
loop_builder.set_entry_point("agent")
loop_builder.add_conditional_edges("agent", looping_should_continue)
loop_builder.add_edge("tools", "agent")

looping_agent = loop_builder.compile()
print("무한루프 에이전트 준비 완료")
print("주의: 다음 셀에서 의도적으로 루프를 실행합니다 (recursion_limit으로 안전하게 종료됨)")

In [ ]:
# 무한루프 에이전트 실행 (recursion_limit=8로 안전하게 종료)
call_count["search"] = 0  # 카운트 초기화

loop_handler = CallbackHandler(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
    session_id="ep06-loop-test",
    tags=["ep06", "debugging", "loop-test"],
)

print("=== 무한루프 에이전트 실행 시작 ===")
try:
    result = looping_agent.invoke(
        {"messages": [HumanMessage(content="인공지능에 대해 검색해줘")]},
        config={
            "callbacks": [loop_handler],
            "recursion_limit": 8,  # 낮게 설정하여 루프 감지
        },
    )
    print("예상치 못한 정상 종료:", result["messages"][-1].content)
except GraphRecursionError as e:
    print(f"GraphRecursionError 발생! (예상된 결과)")
    print(f"총 도구 호출 횟수: {call_count['search']}")
    print("\nLangfuse 대시보드에서 반복 패턴을 확인하세요!")

## 섹션 6. recursion_limit으로 무한루프 방지

`GraphRecursionError`를 캐치하여 graceful한 fallback 응답을 반환합니다.

In [ ]:
def safe_invoke(agent, query: str, handler: CallbackHandler, max_recursion: int = 15):
    """안전한 에이전트 실행 래퍼: GraphRecursionError 처리 포함"""
    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=query)]},
            config={
                "callbacks": [handler],
                "recursion_limit": max_recursion,
            },
        )
        final_answer = result["messages"][-1].content
        status = "success"
    except GraphRecursionError as e:
        print(f"[경고] 무한루프 감지: {e}")
        final_answer = (
            "죄송합니다. 처리 중 복잡도 한도에 도달했습니다. "
            "질문을 더 구체적으로 바꿔서 다시 시도해주세요."
        )
        status = "loop_detected"
    except Exception as e:
        print(f"[오류] 예상치 못한 오류: {e}")
        final_answer = "시스템 오류가 발생했습니다. 잠시 후 다시 시도해주세요."
        status = "error"

    return {"answer": final_answer, "status": status}

# 테스트
call_count["search"] = 0
guard_handler = CallbackHandler(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
    tags=["ep06", "guardrail-test"],
)

result = safe_invoke(looping_agent, "머신러닝이 뭔지 알려줘", guard_handler, max_recursion=6)
print(f"상태: {result['status']}")
print(f"응답: {result['answer']}")

## 섹션 7. Langfuse로 Iteration Count 추적

Langfuse `score` API로 에이전트 반복 횟수를 정량적으로 기록합니다.

In [ ]:
from typing import TypedDict, Annotated
import operator

# 반복 횟수를 추적하는 커스텀 상태
class TrackedState(TypedDict):
    messages: Annotated[list, operator.add]
    iteration_count: int

def tracked_agent_node(state: TrackedState):
    """반복 횟수를 추적하는 에이전트 노드"""
    current_iter = state.get("iteration_count", 0) + 1
    print(f"  [에이전트 반복 #{current_iter}]")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response], "iteration_count": current_iter}

def tracked_should_continue(state: TrackedState):
    last_message = state["messages"][-1]
    # 반복 횟수 초과 시 강제 종료
    if state.get("iteration_count", 0) >= 5:
        print("  [가드레일] 최대 반복 횟수 도달, 강제 종료")
        return END
    if last_message.tool_calls:
        return "tools"
    return END

tracked_builder = StateGraph(TrackedState)
tracked_builder.add_node("agent", tracked_agent_node)
tracked_builder.add_node("tools", ToolNode(tools))
tracked_builder.set_entry_point("agent")
tracked_builder.add_conditional_edges("agent", tracked_should_continue)
tracked_builder.add_edge("tools", "agent")
tracked_agent = tracked_builder.compile()

# 실행 및 iteration count Langfuse 기록
iter_handler = CallbackHandler(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
    tags=["ep06", "iteration-tracking"],
)

result = tracked_agent.invoke(
    {"messages": [HumanMessage(content="파이썬을 검색하고 요약해줘")], "iteration_count": 0},
    config={"callbacks": [iter_handler], "recursion_limit": 20},
)

final_iter = result.get("iteration_count", 0)
print(f"\n총 반복 횟수: {final_iter}")

# Langfuse에 iteration count score 기록
try:
    trace_id = iter_handler.get_trace_id()
    langfuse.score(
        trace_id=trace_id,
        name="iteration_count",
        value=float(final_iter),
        data_type="NUMERIC",
        comment=f"에이전트 반복 횟수: {final_iter}",
    )
    print(f"Langfuse에 iteration_count={final_iter} 기록 완료")
except Exception as e:
    print(f"Langfuse 기록 실패 (Langfuse 미연결 시 정상): {e}")

## 섹션 8. 환각 탐지: LLM 응답 자기일관성 체크

동일한 질문을 여러 번 실행하여 응답 일관성을 측정합니다.
일관성이 낮을수록 환각 가능성이 높습니다.

In [ ]:
import random

def check_self_consistency(
    agent,
    query: str,
    n_runs: int = 3,
    key_fact: str = None,
) -> dict:
    """
    자기일관성 체크: 동일 쿼리를 n_runs회 실행하여 응답 일관성 측정
    key_fact: 응답에 반드시 포함되어야 하는 핵심 사실 (없으면 길이 기반 비교)
    """
    responses = []
    for i in range(n_runs):
        handler = CallbackHandler(
            public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
            secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
            host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            tags=["ep06", "consistency-check", f"run-{i+1}"],
        )
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=query)]},
                config={"callbacks": [handler], "recursion_limit": 10},
            )
            responses.append(result["messages"][-1].content)
        except Exception as e:
            responses.append(f"ERROR: {e}")

    # 일관성 측정: 핵심 사실 포함 비율
    if key_fact:
        contain_fact = sum(1 for r in responses if key_fact.lower() in r.lower())
        consistency_score = contain_fact / len(responses)
    else:
        # 길이 기반 유사도 (간단한 근사)
        lengths = [len(r) for r in responses]
        avg_len = sum(lengths) / len(lengths)
        variance = sum((l - avg_len) ** 2 for l in lengths) / len(lengths)
        # 분산이 낮을수록 일관성이 높음 (0~1로 정규화)
        consistency_score = max(0, 1 - variance / (avg_len ** 2 + 1))

    return {
        "responses": responses,
        "consistency_score": round(consistency_score, 3),
        "is_hallucination_risk": consistency_score < 0.6,
    }

# 자기일관성 테스트
print("자기일관성 체크 실행 중...")
consistency_result = check_self_consistency(
    basic_agent,
    query="파이썬이 무엇인지 한 문장으로 설명해줘",
    n_runs=3,
    key_fact="프로그래밍",
)

print(f"\n일관성 점수: {consistency_result['consistency_score']}")
print(f"환각 위험: {consistency_result['is_hallucination_risk']}")
print("\n각 응답:")
for i, resp in enumerate(consistency_result["responses"], 1):
    print(f"  [{i}] {resp[:100]}...")

## 섹션 9. Langfuse score API로 품질 점수 기록

자기일관성 점수를 Langfuse에 기록하여 시간에 따른 품질 변화를 추적합니다.

In [ ]:
def run_and_score(
    agent,
    query: str,
    session_id: str = "ep06-scored",
    key_fact: str = None,
) -> dict:
    """에이전트 실행 + 품질 점수 자동 기록"""
    handler = CallbackHandler(
        public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
        host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
        session_id=session_id,
        tags=["ep06", "quality-scoring"],
    )

    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=query)]},
            config={"callbacks": [handler], "recursion_limit": 15},
        )
        answer = result["messages"][-1].content
        status = "success"

        # 품질 지표 계산
        has_key_fact = (key_fact.lower() in answer.lower()) if key_fact else True
        answer_length_score = min(1.0, len(answer) / 200)  # 200자 이상이면 1.0
        quality_score = (0.6 * float(has_key_fact)) + (0.4 * answer_length_score)

        # Langfuse score 기록
        trace_id = handler.get_trace_id()
        try:
            langfuse.score(
                trace_id=trace_id,
                name="answer_quality",
                value=round(quality_score, 3),
                data_type="NUMERIC",
                comment=f"핵심사실포함:{has_key_fact}, 길이점수:{answer_length_score:.2f}",
            )
            print(f"Langfuse score 기록: answer_quality={quality_score:.3f}")
        except Exception as e:
            print(f"score 기록 실패: {e}")

    except GraphRecursionError:
        answer = "처리 한도 초과"
        status = "loop_error"
        quality_score = 0.0

    return {"answer": answer, "status": status, "quality_score": quality_score}

# 여러 쿼리로 품질 점수 기록
test_queries = [
    ("파이썬 언어의 특징을 설명해줘", "프로그래밍"),
    ("LangChain이 무엇인지 검색해서 알려줘", "LLM"),
    ("3 * 7 + 10을 계산해줘", "31"),
]

print("품질 점수 기록 테스트")
print("=" * 50)
for query, key_fact in test_queries:
    print(f"\n쿼리: {query}")
    result = run_and_score(basic_agent, query, key_fact=key_fact)
    print(f"상태: {result['status']}, 품질점수: {result['quality_score']:.3f}")
    print(f"응답: {result['answer'][:80]}...")

## 섹션 10. Fallback 전략 구현

에이전트 실패 시 사용자에게 의미 있는 응답을 제공하는 fallback 계층을 구현합니다.

In [ ]:
import time

class AgentWithFallback:
    """fallback 전략이 포함된 에이전트 래퍼"""

    def __init__(self, primary_agent, fallback_llm):
        self.primary_agent = primary_agent
        self.fallback_llm = fallback_llm
        self.langfuse = Langfuse(
            public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
            secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
            host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
        )

    def invoke(self, query: str, session_id: str = "fallback-session") -> dict:
        handler = CallbackHandler(
            public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
            secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
            host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            session_id=session_id,
            tags=["ep06", "fallback-strategy"],
        )

        # 1차: 기본 에이전트 시도
        try:
            start = time.time()
            result = self.primary_agent.invoke(
                {"messages": [HumanMessage(content=query)]},
                config={"callbacks": [handler], "recursion_limit": 10},
            )
            elapsed = time.time() - start
            answer = result["messages"][-1].content
            used_fallback = False
            print(f"기본 에이전트 성공 ({elapsed:.2f}s)")

        except GraphRecursionError:
            print("[Fallback 1] 무한루프 감지 → 직접 LLM 호출로 대체")
            # 2차 fallback: 도구 없이 직접 LLM 응답
            try:
                response = self.fallback_llm.invoke(
                    [HumanMessage(content=f"다음 질문에 간단히 답해주세요: {query}")]
                )
                answer = f"[간소화된 응답] {response.content}"
                used_fallback = True
            except Exception as e2:
                print(f"[Fallback 2] LLM도 실패 → 기본 메시지 반환: {e2}")
                answer = "현재 서비스를 이용할 수 없습니다. 잠시 후 다시 시도해주세요."
                used_fallback = True

        except Exception as e:
            print(f"[Fallback] 예상치 못한 오류: {e}")
            answer = "오류가 발생했습니다. 관리자에게 문의해주세요."
            used_fallback = True

        # fallback 여부를 Langfuse에 기록
        try:
            trace_id = handler.get_trace_id()
            self.langfuse.score(
                trace_id=trace_id,
                name="used_fallback",
                value=1.0 if used_fallback else 0.0,
                data_type="NUMERIC",
                comment="0=정상, 1=fallback 사용",
            )
        except Exception:
            pass

        return {"answer": answer, "used_fallback": used_fallback}

# fallback 에이전트 테스트
fallback_llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
robust_agent = AgentWithFallback(looping_agent, fallback_llm)

print("=== Fallback 전략 테스트 ===")
call_count["search"] = 0
result = robust_agent.invoke("딥러닝에 대해 알려줘", session_id="fallback-test")
print(f"\nFallback 사용: {result['used_fallback']}")
print(f"응답: {result['answer'][:150]}")

## 섹션 11. 종합: 가드레일이 적용된 프로덕션 에이전트 패턴

**[에이전트 무한루프 가드레일 흐름]**
```mermaid
flowchart TD
    A("🤖 1. 기본 에이전트 구축"):::step --> B("☁️ 2. Langfuse 연동"):::step
    B --> C("🔄 3. 무한루프 시나리오 생성"):::test
    C --> D("📊 4. 트레이스 분석"):::test
    D --> E{"🧐 루프 감지됨?"}:::decide
    E -->|Yes| F("🛡️ 5. recursion_limit 설정"):::fix
    E -->|No| G("🔁 더 강한 조건 유도"):::warn
    G --> C
    F --> H("🧪 6. 가드레일 적용 후 재테스트"):::fix
    H --> I("👀 7. Langfuse 대시보드 확인"):::dash
    I --> J("⚖️ 8. 환각(Hallucination) 평가 추가"):::dash
    J --> K("🚀 9. 프로덕션 준비 완료"):::pass
    classDef step fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef test fill:#fff3e0,stroke:#fb8c00,stroke-width:2px,color:#000
    classDef decide fill:#ffe0b2,stroke:#f57c00,stroke-width:2px,font-weight:bold,color:#000
    classDef warn fill:#ffcdd2,stroke:#e53935,stroke-width:2px,color:#000
    classDef fix fill:#e8f5e9,stroke:#43a047,stroke-width:2px,color:#000
    classDef dash fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px,color:#000
    classDef pass fill:#c8e6c9,stroke:#2e7d32,stroke-width:3px,font-weight:bold,color:#000
```

지금까지 배운 모든 패턴을 하나로 통합한 프로덕션 수준의 에이전트를 구성합니다.

In [ ]:
from collections import deque

class ProductionAgent:
    """
    프로덕션 수준의 가드레일이 적용된 에이전트:
    - recursion_limit: 무한루프 방지
    - 도구 중복 호출 감지
    - Langfuse 자동 트레이싱
    - fallback 응답
    - iteration count 기록
    """

    def __init__(
        self,
        agent,
        fallback_llm,
        max_recursion: int = 15,
        max_duplicate_calls: int = 3,
    ):
        self.agent = agent
        self.fallback_llm = fallback_llm
        self.max_recursion = max_recursion
        self.max_duplicate_calls = max_duplicate_calls
        self.langfuse = Langfuse(
            public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
            secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
            host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
        )

    def invoke(self, query: str, user_id: str = "anonymous") -> dict:
        handler = CallbackHandler(
            public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
            secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
            host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            user_id=user_id,
            tags=["production", "ep06"],
        )

        metrics = {
            "status": "unknown",
            "used_fallback": False,
            "latency": 0.0,
        }

        start_time = time.time()
        try:
            result = self.agent.invoke(
                {"messages": [HumanMessage(content=query)]},
                config={
                    "callbacks": [handler],
                    "recursion_limit": self.max_recursion,
                },
            )
            answer = result["messages"][-1].content
            metrics["status"] = "success"

        except GraphRecursionError:
            print("[가드레일] 무한루프 감지 → fallback 활성화")
            response = self.fallback_llm.invoke(
                [HumanMessage(content=f"간단히 답해주세요: {query}")]
            )
            answer = response.content
            metrics["status"] = "loop_fallback"
            metrics["used_fallback"] = True

        except Exception as e:
            answer = "서비스 오류. 잠시 후 재시도해주세요."
            metrics["status"] = "error"
            metrics["used_fallback"] = True

        metrics["latency"] = round(time.time() - start_time, 3)

        # Langfuse에 메트릭 기록
        try:
            trace_id = handler.get_trace_id()
            self.langfuse.score(
                trace_id=trace_id,
                name="execution_status",
                value=1.0 if metrics["status"] == "success" else 0.0,
                comment=metrics["status"],
            )
        except Exception:
            pass

        return {"answer": answer, "metrics": metrics}

# 프로덕션 에이전트 테스트
prod_agent = ProductionAgent(
    agent=basic_agent,
    fallback_llm=ChatAnthropic(model="claude-haiku-4-5", temperature=0),
    max_recursion=15,
)

print("=== 프로덕션 에이전트 테스트 ===")
test_result = prod_agent.invoke(
    "LangChain이 무엇인지 검색하고 간단히 설명해줘",
    user_id="ep06-student",
)
print(f"\n상태: {test_result['metrics']['status']}")
print(f"응답시간: {test_result['metrics']['latency']}s")
print(f"Fallback 사용: {test_result['metrics']['used_fallback']}")
print(f"\n응답:\n{test_result['answer']}")

## 섹션 12. 정리

이 노트북에서 배운 것:

| 주제 | 핵심 내용 |
|------|----------|
| Langfuse 연동 | `CallbackHandler`로 자동 트레이싱 |
| 무한루프 탐지 | `GraphRecursionError` + `recursion_limit` |
| Iteration 추적 | 커스텀 State + Langfuse score API |
| 환각 탐지 | 자기일관성 체크 (n_runs) |
| 품질 점수 기록 | `langfuse.score()` + `data_type="NUMERIC"` |
| Fallback 전략 | try/except + 직접 LLM 호출 |
| 프로덕션 패턴 | 위 모든 요소 통합 |

## Exercise

### Exercise 1: 무한루프 에이전트 만들고 Langfuse로 탐지

**목표**: 의도적으로 무한루프에 빠지는 에이전트를 만들고, Langfuse 트레이스에서 루프 패턴을 확인한 뒤 가드레일로 수정한다.

**단계**:
1. 항상 `"결과 불충분, 재검색 필요"`를 반환하는 도구 `unstable_search` 구현
2. 이 도구를 사용하는 에이전트를 `recursion_limit=50`으로 실행 (의도적 루프)
3. Langfuse 대시보드에서 반복 호출 패턴 시각화 확인
4. `recursion_limit=10`으로 수정하고, `GraphRecursionError` 발생 시 fallback 응답 반환
5. 루프 감지 여부를 `loop_detected` score (0 또는 1)로 Langfuse에 기록

In [ ]:
# Exercise 1: 여기에 코드를 작성하세요

# Step 1: unstable_search 도구 구현
# TODO

# Step 2: 루프 에이전트 구성 및 실행
# TODO

# Step 3: Langfuse 대시보드 URL 출력
# TODO

# Step 4: recursion_limit=10 + fallback 적용
# TODO

# Step 5: loop_detected score 기록
# TODO

### Exercise 2: LangGraph 가드레일 적용 전후 비교

**목표**: 동일한 에이전트에 가드레일을 단계적으로 적용하여 Langfuse에서 전후 지표를 비교한다.

**단계**:
1. `basic_agent`로 5개 테스트 쿼리 실행 → `tags=["no-guardrail"]`
2. 각 실행에 `answer_quality` score 기록 (핵심 키워드 포함 여부 기준)
3. `ProductionAgent`(가드레일 포함)로 동일 5개 쿼리 실행 → `tags=["with-guardrail"]`
4. 동일하게 `answer_quality` score 기록
5. 두 그룹의 평균 score와 오류율을 표로 출력

**힌트**: `tags`는 Langfuse 대시보드에서 그룹 필터링에 사용됩니다.

In [ ]:
# Exercise 2: 여기에 코드를 작성하세요

test_queries_ex2 = [
    ("파이썬의 특징을 설명해줘", "인터프리터"),
    ("LangChain이 뭔지 검색해줘", "프레임워크"),
    ("10 * 10 + 5를 계산해줘", "105"),
    ("머신러닝과 딥러닝의 차이를 설명해줘", "신경망"),
    ("프로그래밍 언어를 배우는 방법을 알려줘", "실습"),
]

# Step 1~2: 가드레일 없는 에이전트 실행
# TODO

# Step 3~4: 가드레일 있는 에이전트 실행
# TODO

# Step 5: 비교 결과 출력
# TODO